# YOLOv8 fine-tuning on Colab (Football Analysis Tool)

**Why Colab:** laptop has Intel Iris Xe iGPU — fine for inference, too slow / VRAM-limited for training. Colab's free T4 finishes a YOLOv8m run on ~500 labeled frames in ~20–30 min.

## Runtime
`Runtime → Change runtime type → GPU (T4)`

## Expected inputs
1. A zipped YOLO-format dataset (`dataset.zip` containing `images/{train,val}` + `labels/{train,val}` + `data.yaml`).
2. Either upload via the file browser or mount Google Drive.

In [ ]:
# 1. Verify GPU
!nvidia-smi

In [ ]:
# 2. Install
!pip install -q ultralytics supervision

In [ ]:
# 3. (Option A) Pull Roboflow's public football-players-detection dataset as a starting point.
#    Replace WORKSPACE/PROJECT/VERSION with the exact values from the Roboflow dataset page.
#    This is the dataset we transfer-learn FROM; we'll later fine-tune on our own clips.
#
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_KEY")
# project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
# version = project.version(12)
# dataset = version.download("yolov8")
# DATA_YAML = f"{dataset.location}/data.yaml"

# 3. (Option B) Upload our own dataset.zip (from Label Studio → YOLO export) and unzip.
from google.colab import files
uploaded = files.upload()   # pick dataset.zip
!unzip -q dataset.zip -d /content/dataset
DATA_YAML = "/content/dataset/data.yaml"
print("Using:", DATA_YAML)

In [ ]:
# 4. Sanity-check data.yaml
!cat {DATA_YAML}
# Expected classes (order matters, must match Label Studio export):
#   0: player
#   1: goalkeeper
#   2: referee
#   3: ball

In [ ]:
# 5. Train. Start from YOLOv8m pretrained on COCO.
#    - imgsz=1280 matters for tiny-ball detection.
#    - 50 epochs is a sensible default; watch val loss in results.png and stop early if plateaued.
from ultralytics import YOLO

model = YOLO("yolov8m.pt")
results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=1280,
    batch=8,
    patience=10,
    project="runs/football",
    name="yolov8m-ft",
)

In [ ]:
# 6. Evaluate on val set
metrics = model.val()
print(metrics.box.map, metrics.box.map50, metrics.box.map75)

In [ ]:
# 7. Download the fine-tuned weights — drop into models/best.pt in the project.
from google.colab import files
files.download("runs/football/yolov8m-ft/weights/best.pt")